# Notebook 11 — Figures, contrôle STT et export des résultats (CPU, ~30 min)

Remplace les notebooks 3, 5 et 6 pour la version finale (labels v2).

**Réglages :** *Accelerator* **None** ; *Internet* **On** (téléchargement de `ptbxl_database.csv`).
*Inputs* : le dataset du code, la sortie du notebook 1 (cache) et la **sortie finale du notebook 9**
(celle qui affiche « FILE D'ATTENTE TERMINÉE »).

À la fin, téléchargez **un seul fichier** : `resultats_finaux.zip` (lien affiché en bas).

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
import ecgfusion, json, zipfile
assert ecgfusion.__version__ >= "2026.11.20", "Ancienne version du code en Input"
runs = {os.path.basename(os.path.dirname(os.path.dirname(p))): os.path.dirname(os.path.dirname(p))
        for p in glob.glob("/kaggle/input/**/runs/L2_*/eval/summary.csv", recursive=True)}
print(len(runs), "runs :", sorted(runs))
assert len(runs) == 13, "Il faut les 13 runs L2_* : ajoutez la sortie FINALE du notebook 9 en Input."
OUT = "/kaggle/working/final"
os.makedirs(OUT, exist_ok=True)
FINAL_RUNS = [runs[n] for n in ["L2_final_s42", "L2_final_s1", "L2_final_s2"]]

In [ ]:
# 1) Figures de l'article (modèle final, graine 42) : architecture, effets par pathologie, gate, calibration
!python scripts/fig_architecture.py --out {OUT}/figures
!python scripts/make_figures.py --run {FINAL_RUNS[0]} --out {OUT}/figures

In [ ]:
# 2) Contrôle STT_ISCH en dérivation I (3 graines du modèle final)
metas = sorted(glob.glob("/kaggle/input/**/cache/ptbxl/meta.csv", recursive=True))
PTB = os.path.dirname(metas[0]) if metas else None
cmd = ["python", "scripts/stt_lead_check.py", "--runs", *FINAL_RUNS, "--out", f"{OUT}/stt_check"]
if PTB:
    cmd += ["--ptbxl_cache", PTB]
import shlex
!{shlex.join(cmd)}

In [ ]:
# 3) Export : tous les tableaux (CSV/JSON) des 13 runs + figures + contrôle STT, sans les prédictions brutes
keep_ext = (".csv", ".json", ".yaml", ".pdf", ".png", ".done")
zpath = "/kaggle/working/resultats_finaux.zip"
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for name, r in sorted(runs.items()):
        for f in glob.glob(f"{r}/**/*", recursive=True):
            if os.path.isfile(f) and f.endswith(keep_ext) and "/reliability_" not in f:
                z.write(f, f"runs/{name}/" + os.path.relpath(f, r)); n += 1
    for f in glob.glob(f"{OUT}/**/*", recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, OUT)); n += 1
    for f in glob.glob("/kaggle/input/**/lsas_codes.csv", recursive=True)[:1]:
        z.write(f, "lsas_codes.csv"); n += 1
print(f"{n} fichiers -> {zpath} ({os.path.getsize(zpath) / 1e6:.1f} Mo)")
os.chdir("/kaggle/working")
from IPython.display import FileLink, display
display(FileLink("resultats_finaux.zip"))